In [1]:
import pandas as pd
import numpy as np

# Use RidgeCV for cross-validation and StandardScaler for data scaling
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# =========================
# Load cleaned dataset
# =========================
df_model = pd.read_csv("data_clean_2.csv")

# Convert date column
df_model["MthCalDt"] = pd.to_datetime(df_model["MthCalDt"])

print("Original df_model shape:", df_model.shape)
print("Original date range:", df_model["MthCalDt"].min(), "to", df_model["MthCalDt"].max())

display(df_model.head())

Original df_model shape: (1501178, 24)
Original date range: 1998-04-30 00:00:00 to 2025-11-28 00:00:00


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthCap,MthRet,ShrOut,MthPrc_Abs,DateKey,Mkt-RF,SMB,HML,RMW,CMA,RF,ExcessRet,y_next,ret_lag1,ret_lag2,ret_lag3,history_len,log_size
0,10001,36720410,EWST,7953,1998-04-30,8.8125,21158.81,0.007143,2401,8.8125,199804,0.0074,0.0001,0.0103,-0.0168,-0.0036,0.0043,0.002843,-0.018184,-0.012702,-0.010844,-0.004300,36,9.959812
1,10001,36720410,EWST,7953,1998-05-29,8.6875,20858.69,-0.014184,2401,8.6875,199805,-0.0305,-0.0300,0.0341,0.0103,0.0247,0.0040,-0.018184,0.001754,0.002843,-0.012702,-0.010844,37,9.945526
2,10001,36720410,EWST,7953,1998-06-30,8.6250,20725.88,0.005854,2403,8.6250,199806,0.0321,-0.0361,-0.0203,-0.0026,-0.0303,0.0041,0.001754,0.010493,-0.018184,0.002843,-0.012702,38,9.939138
3,10001,36720410,EWST,7953,1998-07-31,8.7500,21026.25,0.014493,2403,8.7500,199807,-0.0242,-0.0520,-0.0183,0.0164,0.0076,0.0040,0.010493,-0.004300,0.001754,-0.018184,0.002843,39,9.953527
4,10001,36720410,EWST,7953,1998-08-31,8.7500,21026.25,0.000000,2403,8.7500,199808,-0.1605,-0.0497,0.0357,0.0335,0.0578,0.0043,-0.004300,0.065686,0.010493,0.001754,-0.018184,40,9.953527


In [2]:
# =========================
# Feature Engineering
# =========================

# 1. Create squared terms for historical returns (Non-linear momentum)
for lag in ['ret_lag1', 'ret_lag2', 'ret_lag3']:
    df_model[f'{lag}_sq'] = df_model[lag] ** 2

# 2. Create interaction terms (Macro factors * Firm Size)
macro_factors = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
for factor in macro_factors:
    df_model[f'{factor}_x_size'] = df_model[factor] * df_model['log_size']

print("Enhanced df shape:", df_model.shape)

# Define Base Feature columns
base_features = [
    "Mkt-RF", "SMB", "HML", "RMW", "CMA",
    "ret_lag1", "ret_lag2", "ret_lag3", "log_size"
]

# Append new features
new_features = [f'{lag}_sq' for lag in ['ret_lag1', 'ret_lag2', 'ret_lag3']] + \
               [f'{factor}_x_size' for factor in macro_factors]

feature_cols = base_features + new_features
target_col = "y_next"

# All available months
all_months = sorted(df_model["DateKey"].unique())

print("Number of months:", len(all_months))
print("Number of features:", len(feature_cols))
print("Enhanced Features:", feature_cols)

Enhanced df shape: (1501178, 32)
Number of months: 332
Number of features: 17
Enhanced Features: ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'ret_lag1', 'ret_lag2', 'ret_lag3', 'log_size', 'ret_lag1_sq', 'ret_lag2_sq', 'ret_lag3_sq', 'Mkt-RF_x_size', 'SMB_x_size', 'HML_x_size', 'RMW_x_size', 'CMA_x_size']


In [3]:
# =========================
# Rolling 36-month Enhanced OLS (RidgeCV)
# =========================

results = []
months = all_months

# Define a range of penalty terms (alphas) to test using log scale
alphas_to_test = np.logspace(-2, 4, 13)
print(f"Testing the following alpha candidates:\n{alphas_to_test}\n")

# Initialize RidgeCV (Leave-One-Out Cross-Validation by default)
model = RidgeCV(alphas=alphas_to_test)

# Track the optimal alpha chosen each month
optimal_alphas = []

for i in range(36, len(months)):
    month = months[i]

    # Previous 36 months for training
    train_months = months[i - 36 : i]

    train_data = df_model[df_model["DateKey"].isin(train_months)]
    test_data = df_model[df_model["DateKey"] == month]

    if len(train_data) == 0 or len(test_data) == 0:
        continue

    X_train = train_data[feature_cols]
    y_train = train_data[target_col]
    X_test = test_data[feature_cols]

    # Feature scaling (Strictly fit on train data only to avoid data leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train and automatically select the best alpha
    model.fit(X_train_scaled, y_train)

    # Predict next month's return
    preds = model.predict(X_test_scaled)

    # Store predictions
    temp = test_data.copy()
    temp["predicted_return"] = preds
    results.append(temp)

    # Save the best alpha chosen for the current window
    optimal_alphas.append({'DateKey': month, 'best_alpha': model.alpha_})

# Combine all predictions
pred_df = pd.concat(results)
alpha_history = pd.DataFrame(optimal_alphas)

print("Prediction dataset shape:", pred_df.shape)
display(pred_df[['DateKey', 'Ticker', 'y_next', 'predicted_return']].head())

print("\nSample of optimal alphas chosen by RidgeCV (Recent Months):")
display(alpha_history.tail(10))

Testing the following alpha candidates:
[1.00000000e-02 3.16227766e-02 1.00000000e-01 3.16227766e-01
 1.00000000e+00 3.16227766e+00 1.00000000e+01 3.16227766e+01
 1.00000000e+02 3.16227766e+02 1.00000000e+03 3.16227766e+03
 1.00000000e+04]



/Users/haojingcheng/PycharmProjects/586/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/haojingcheng/PycharmProjects/586/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/haojingcheng/PycharmProjects/586/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/haojingcheng/PycharmProjects/586/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/haojingcheng/PycharmProjects/586/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/haojingcheng/PycharmProjects/586/.venv/lib/python3.10/site-packages/sklearn/linear_model/_

Prediction dataset shape: (1351094, 33)


,DateKey,Ticker,y_next,predicted_return
36,200104,EWST,0.094236,-0.000708
268,200104,SABC,-0.036592,0.014666
432,200104,SCTT,-0.032136,-0.004956
485,200104,AEPI,0.114904,-0.008161
710,200104,JJSF,0.133208,-0.005892



Sample of optimal alphas chosen by RidgeCV (Recent Months):


,DateKey,best_alpha
286,202502,100.000000
287,202503,100.000000
288,202504,31.622777
289,202505,31.622777
290,202506,31.622777
291,202507,31.622777
292,202508,100.000000
293,202509,31.622777
294,202510,31.622777
295,202511,31.622777


In [4]:
# =========================
# Evaluate model accuracy (POST-2022 ONLY)
# =========================

pred_df_post2022 = pred_df[pred_df["MthCalDt"] >= "2022-01-01"].copy()

mse = mean_squared_error(pred_df_post2022["y_next"], pred_df_post2022["predicted_return"])
mae = mean_absolute_error(pred_df_post2022["y_next"], pred_df_post2022["predicted_return"])

print("Enhanced OLS (RidgeCV) Performance (Post-2022)")
print(f"MSE: {mse:.6f}")
print(f"MAE: {mae:.6f}")

print("\nPrediction Statistics:")
display(pred_df_post2022["predicted_return"].describe())

Enhanced OLS (RidgeCV) Performance (Post-2022)
MSE: 0.010160
MAE: 0.070280

Prediction Statistics:


count    252411.000000
mean          0.008343
std           0.018102
min          -0.085834
25%          -0.003106
50%           0.008371
75%           0.018899
max           0.144528
Name: predicted_return, dtype: float64

In [5]:
# =========================
# Rank stocks each month and select Top 6
# =========================

pred_df_post2022["rank"] = pred_df_post2022.groupby("DateKey")["predicted_return"].rank(ascending=False)

top6_enhanced = pred_df_post2022[pred_df_post2022["rank"] <= 6].copy()

print("Top-6 dataset shape:", top6_enhanced.shape)
display(top6_enhanced[['DateKey', 'Ticker', 'y_next', 'predicted_return', 'rank']].head())

# =========================
# Save Enhanced Selections
# =========================
# Ensure the path is correct based on your folder structure
top6_enhanced.to_csv("enhanced_ols_ridgecv_top6_by_month_post2022.csv", index=False)
print("Enhanced OLS (RidgeCV) Top-6 selections saved successfully.")

Top-6 dataset shape: (282, 34)


,DateKey,Ticker,y_next,predicted_return,rank
69228,202201,DQ,0.195414,0.082690,5.0
175719,202201,RDUS,0.097625,0.088548,4.0
191489,202201,LC,-0.009595,0.089952,3.0
352868,202201,CAR,0.041208,0.106411,1.0
505003,202201,YELL,-0.136973,0.081863,6.0


Enhanced OLS (RidgeCV) Top-6 selections saved successfully.
